# Lesson 20 — Practice: Project Structure

Work in `lessons/lesson-20-structure/` (point a terminal at that folder).

Before you start, read `lesson.md` — every construct used in these exercises is
taught there. Provided trees are read-only instruments unless a spec says fix
it: `launchlab/`, `importlab/`, `vend/GLOBAL_FROZEN.txt`, `flat_shop/`,
`messy_app/` (buggy), `report_tool_flat/`, `ledger_flat/`, plus the suites
`test_messy.py` / `test_restructure.py` (do NOT modify) and blank
`solution.py`. No installs needed — stdlib only.


## E1 — Predict sys.path[0] under two launches

## Intent
**See the launch context: predict which folder Python searches first before you run anything.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
`launchlab/src/shop/main.py` prints `sys.path[0]` — it is a measurement
instrument, not an app. Predict its output for two launches, then let the
Driver cell confront your predictions with reality.

**File tree**

```
launchlab/src/shop/main.py   (provided — read it, do NOT modify)
paths.py                     (you create)
```

**Interface contract**
`paths.py` defines two strings, tails with forward slashes starting at the
lesson folder:
`P_SCRIPT` — predicted output of `python launchlab/src/shop/main.py` run
with cwd = the lesson folder.
`P_MODULE` — predicted output of `python -m shop.main` run with cwd =
`launchlab/src`.

**Constraints**
- **Predict first — write both strings before running anything.**
- **Tails only** (`lesson-20-structure/...`), forward slashes — absolute paths vary by machine.
- Driver imports `import paths as p` and runs both launches live.

**Coverage**
Three launches + `sys.path[0]` table → lesson.md Core Concepts 1/2.

**Expected** (Driver cell):
```
2/2 launch predictions correct
```


In [1]:
import subprocess
import sys
from pathlib import Path

import paths as p

lesson = Path.cwd()


def tail(path):
    parts = Path(path).parts
    i = list(parts).index("launchlab")
    return "/".join(parts[i - 1:])


r1 = subprocess.run(
    [sys.executable, "launchlab/src/shop/main.py"],
    capture_output=True, text=True, cwd=lesson,
)
r2 = subprocess.run(
    [sys.executable, "-m", "shop.main"],
    capture_output=True, text=True, cwd=lesson / "launchlab" / "src",
)
got = [tail(r1.stdout.strip()), tail(r2.stdout.strip())]
want = [p.P_SCRIPT, p.P_MODULE]
print(f"{sum(a == b for a, b in zip(got, want))}/2 launch predictions correct")

2/2 launch predictions correct


## E2 — Judge which imports survive which launch

## Intent
**Choose imports from context: fill the 6-cell truth table of file × launch without executing it.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
`importlab/` holds three files with three import styles. For each of the six
(file, launch) cells below, predict `True` (runs, prints `1`) or `False`
(crashes) — from reasoning, then let the Driver cell run all six.

**File tree**

```
importlab/a.py  importlab/pkg/use.py  importlab/pkg/rel.py   (provided — read them)
imports.py                                                 (you create)
```

**Interface contract**
`imports.py` defines `WORKS`, a dict keyed exactly:
`("a.py", "script")`, `("a.py", "module")`, `("pkg/use.py", "script")`,
`("pkg/use.py", "module")`, `("pkg/rel.py", "script")`,
`("pkg/rel.py", "module")` — `"script"` means `python <file>`,
`"module"` means `python -m <dotted>`, both with cwd = `importlab/`.

**Constraints**
- **Reason first — fill the table before running anything; the Driver is the judge, not the worksheet.**
- Driver imports `import imports as im` and executes every cell live.

**Coverage**
Absolute vs relative by launch context → lesson.md Core Concept 3.

**Expected** (Driver cell):
```
6/6 import predictions correct
```


In [2]:
import subprocess
import sys
from pathlib import Path

import imports as im

lab = Path.cwd() / "importlab"
cells = [
    (("a.py", "script"), ["a.py"]),
    (("a.py", "module"), ["-m", "a"]),
    (("pkg/use.py", "script"), ["pkg/use.py"]),
    (("pkg/use.py", "module"), ["-m", "pkg.use"]),
    (("pkg/rel.py", "script"), ["pkg/rel.py"]),
    (("pkg/rel.py", "module"), ["-m", "pkg.rel"]),
]
hits = 0
for key, args in cells:
    r = subprocess.run(
        [sys.executable, *args], capture_output=True, text=True, cwd=lab
    )
    works = r.returncode == 0 and r.stdout.strip() == "1"
    hits += works == im.WORKS[key]
print(f"{hits}/6 import predictions correct")

6/6 import predictions correct


## E3 — Triage a global freeze into honest supporting files

## Intent
**Declare, don't dump: hand-write the two files that describe this project's real dependencies and exclusions.**

**Type:** REAL-WORLD — Easy — writes 2 files

**Task**
`vend/GLOBAL_FROZEN.txt` is a laptop dump (8 lines: `requests`, `pytest`,
`tensorflow`, `torch`, ...). This lesson's exercises import exactly two
third-party packages. Write the honest versions.

**File tree**

```
vend/GLOBAL_FROZEN.txt    (provided — triage source, do NOT copy it)
vend/requirements.txt     (you create)
vend/.gitignore           (you create)
```

**Interface contract**
`vend/requirements.txt` is byte-exactly:
```
requests
pytest
```
`vend/.gitignore` is byte-exactly:
```
.venv/
__pycache__/
*.csv
```

**Constraints**
- **requirements lists what YOU directly import — nothing else; `.gitignore` excludes envs, caches, and data files.**
- Trailing newline on both files; driver compares exact bytes.

**Coverage**
Minimal requirements + honest `.gitignore` → lesson.md Core Concept 5
(WP2 + L19 scars as anti-examples).

**Expected** (Driver cell):
```
requirements OK
gitignore OK
```


In [3]:
from pathlib import Path

req = Path("vend/requirements.txt").read_text(encoding="utf-8")
ign = Path("vend/.gitignore").read_text(encoding="utf-8")
print("requirements", "OK" if req == "requests\npytest\n" else "MISMATCH")
print("gitignore", "OK" if ign == ".venv/\n__pycache__/\n*.csv\n" else "MISMATCH")

requirements OK
gitignore OK


## M1 — Convert a flat pile into a package

## Intent
**Rehouse working code: lift `flat_shop/` into an importable package with a `-m` entry, behavior byte-identical.**

**Type:** REAL-WORLD — Medium — writes 1 package

**Task**
`flat_shop/` works — but only from inside its own folder (`from carts
import ...`, `python run.py`). Rebuild it as `shop/` so it runs from the
project root under both entry forms. Read the flat sources first and
preserve the behavior exactly.

**File tree**

```
flat_shop/carts.py  flat_shop/report.py  flat_shop/run.py   (provided — read only)
shop/__init__.py  shop/carts.py  shop/report.py  shop/main.py   (you create)
```

**Interface contract**
`shop/carts.py` keeps `CARTS` + `line_total(items)`.
`shop/report.py` keeps `summary()` returning exactly
`cart: 3 items, total 7.50`.
`shop/main.py` uses absolute imports (`from shop.report import ...`),
prints `summary()` with no argv, and contains **no `sys.path` manipulation**.

**Constraints**
- **Entry forms checked: `python -m shop.main` AND `PYTHONPATH=<lesson> python shop/main.py` — both print the identical line.**
- `flat_shop/` stays untouched; the behavior string is pinned below.

**Coverage**
Flat → package + absolute-in-entry + `-m` default → lesson.md Core
Concepts 1/2/3/4.

**Expected** (Driver cell):
```
cart: 3 items, total 7.50
cart: 3 items, total 7.50
```


In [4]:
import os
import subprocess
import sys
from pathlib import Path

lesson = Path.cwd()
r1 = subprocess.run(
    [sys.executable, "-m", "shop.main"],
    capture_output=True, text=True, cwd=lesson,
)
print(r1.stdout.strip())
env = {**os.environ, "PYTHONPATH": str(lesson)}
r2 = subprocess.run(
    [sys.executable, "shop/main.py"],
    capture_output=True, text=True, cwd=lesson, env=env,
)
print(r2.stdout.strip())

cart: 3 items, total 7.50
cart: 3 items, total 7.50


## M2 — Harden the entry: argv, guards, foreign-cwd proof

## Intent
**Prove the entry travels: validate argv, guard the loader, and run green from a foreign directory.**

**Type:** REAL-WORLD — Medium — edits 1 package

**Task**
Extend `shop/main.py`: an optional positional CSV path (`price` column),
a loader that rejects bad paths before touching the disk, and proof the
whole thing works with cwd anywhere.

**File tree**

```
shop/main.py    (you edit — add `load_basket` + argv handling)
```

**Interface contract**
`load_basket(path: str) -> list[float]` — reads the `price` column;
**non-`str` or empty path raises `ValueError(f"bad basket path: {path!r}")`
before any file access.**
`main(argv)`: `len(argv) > 2` → `usage: python -m shop.main [basket.csv]`
on **stderr**, exit **1**; one arg → basket file summary
`cart: N items, total D.DD`; no args → the M1 `summary()` line.

**Constraints**
- **The guard uses `or`, and the isinstance check comes first.**
- Driver runs from a temp cwd with `PYTHONPATH=<lesson>` — your entry must not depend on where it stands.

**Coverage**
Positional argv + usage/exit-1 (Stage-1 retrieval) + loader guard →
lesson.md Core Concept 4 + Engineering Notes.

**Expected** (Driver cell):
```
cart: 2 items, total 3.00
usage exit: 1
```


In [5]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

lesson = Path.cwd()
with tempfile.TemporaryDirectory() as td:
    tmp = Path(td)
    (tmp / "basket.csv").write_text("price\n1.00\n2.00\n", encoding="utf-8")
    env = {**os.environ, "PYTHONPATH": str(lesson)}
    r1 = subprocess.run(
        [sys.executable, "-m", "shop.main", "basket.csv"],
        cwd=tmp, capture_output=True, text=True, env=env,
    )
    print(r1.stdout.strip())
    r2 = subprocess.run(
        [sys.executable, "-m", "shop.main", "a", "b"],
        cwd=tmp, capture_output=True, text=True, env=env,
    )
    print("usage exit:", r2.returncode)

cart: 2 items, total 3.00
usage exit: 1


## M3 — Pin the package with tests + conftest

## Intent
**Make the suite guard the structure: fixtures in conftest, three pinned tests, no path hacks.**

**Type:** REAL-WORLD — Medium — writes 1 test dir

**Task**
Give `shop/` the test layout it deserves: shared fixture in
`tests/conftest.py`, three exactly-named tests in `tests/test_shop.py`.
If `pytest tests/` cannot import your package unaided, fix the layout —
never the test path.

**File tree**

```
tests/conftest.py     (you create — `morning_basket` fixture: [2.50, 1.80, 3.20])
tests/test_shop.py    (you create — exactly `test_total`, `test_count`, `test_empty`)
```

**Interface contract**
`test_total`: `line_total(morning_basket) == pytest.approx(7.50)`.
`test_count`: `len(morning_basket) == 3`.
`test_empty`: `line_total([]) == 0.0` (degenerate edge, L8 retrieval).

**Constraints**
- **Exactly these three test functions; the fixture lives in conftest, not duplicated per test.**
- **No `sys.path` manipulation anywhere under `tests/`.**
- Driver runs `pytest tests/test_shop.py -q` and pins the count.

**Coverage**
`tests/` + `conftest` + `approx` + empty edge → lesson.md Core Concept 6
(L17 retrieval).

**Expected** (Driver cell): `3 passed`.


In [6]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_shop.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr)
_m = re.search(r"(\d+ passed)", _out)
print(_m.group(1) if _m else _out[-500:])

3 passed


## H1 — Debug drill: four structural bugs in messy_app

## Intent
**Diagnose structure like code: fix four layout bugs traceback-first, each mapping to a Pitfall.**

**Type:** DEBUG — Hard — edits 1 tree

**Task**
`messy_app/` runs — but only from one folder, with a hack, noisily, and
blind to its own data file. Find and fix all four so the suite goes
green. The test file is correct — you may read it but must not modify it.

**File tree**

```
messy_app/__init__.py  messy_app/data.py  messy_app/prices.csv
messy_app/app.py  messy_app/main.py   (provided, buggy — YOU fix them)
test_messy.py                         (provided — do NOT modify)
```

**Hints**
- Bug 1: importing your entry mutates a global interpreter list. Which
  statement does it — and which launch truth makes it unnecessary?
- Bug 2: importing the package prints. Which file runs at import time,
  and what should it contain instead?
- Bug 3: a leading dot works under `-m` and dies as a script. Which
  file, and which import style survives both?
- Bug 4: the data opens from here-only. Which call assumes your cwd —
  and what does the module know (`__file__`) that you don't?

**Bug report** (fill in as you go — see the next cell).

**Coverage**
Path hacks + noisy init + dot-in-script + cwd-anchored data → lesson.md
Pitfalls 1–4 + Core Concepts 2/3/7; protocol → `notes/DEBUGGING.md`.

**Expected** (Driver cell): `4 passed` — the final state after all four fixes.


### H1 bug report — blank write-in (fill each row as you fix it, never after)

```
# | Expected vs Observed                         | Root cause | Fix
--|----------------------------------------------|------------|----
1 |                                              |            |
2 |                                              |            |
3 |                                              |            |
4 |                                              |            |
```


In [7]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_messy.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr)
_m = re.search(r"(\d+ passed)", _out)
print(_m.group(1) if _m else _out[-500:])

4 passed


## H2 — Restructure the flat patient end-to-end

## Intent
**Deliver a project, not a pile: package + entry + README around real behavior that must not change.**

**Type:** REAL-WORLD — Hard — writes 1 package + docs

**Task**
`report_tool_flat/` computes correctly and launches brittlely (`from load
import ...`, `python run.py` from its own dir only). Rehouse it as
`report_tool/` with a documented entry — totals identical, launch robust.

**File tree**

```
report_tool_flat/load.py  compute.py  write.py  run.py  grades.csv   (provided — read only)
report_tool/__init__.py  load.py  compute.py  write.py  main.py      (you create)
report_tool/README.md                                                (you create)
test_restructure.py                                                  (provided — do NOT modify)
```

**Interface contract**
`summarize(rows)` keeps `.get(team, 0.0)` accumulation (L11 retrieval —
do not "simplify" it).
`python -m report_tool.main <grades.csv>` writes `summary.csv` next to
the input and prints exactly `Summary (2 teams, 6 rows)`; wrong arity →
usage on **stderr**, exit **1**.
`README.md` names the entry command `python -m report_tool.main`.

**Constraints**
- **Absolute imports, `__init__.py` silent, no path hacks — the suite checks all three.**
- **Behavior is frozen: the flat `run.py` output and your entry output agree byte-for-byte.**

**Coverage**
Full rehouse + frozen behavior + README run-commands → lesson.md Core
Concepts 1/3/4/5.

**Expected** (Driver cell): `4 passed`.


In [8]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_restructure.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr)
_m = re.search(r"(\d+ passed)", _out)
print(_m.group(1) if _m else _out[-500:])

4 passed


## Challenge — Build the ledger project from the flat patient

## Intent
**Ship the whole discipline: package + tests + supporting files around merge semantics that must survive the move.**

**Type:** REAL-WORLD — Challenge — writes 1 project

**Task**
`ledger_flat/` merges wrong (cached-only items vanish), writes
Windows-hostile CSV (missing `newline=""`), launches with a path hack,
and takes bare argv without validation. Build `ledger/` — the neutral
mini-project — fixing all four while preserving the file roles.

**File tree**

```
ledger_flat/load.py  compute.py  write.py  run.py  feed.json   (provided — read only)
ledger/__init__.py  load.py  compute.py  write.py  main.py     (you create)
ledger/tests/test_merge.py                                      (you create — exactly the 3 names below)
ledger/requirements.txt  ledger/.gitignore  ledger/README.md   (you create)
```

**Interface contract**
`merge(network, cached) -> (items, fresh, kept)`: **every network id
wins; cached-only items are kept; output sorted by id.**
`python -m ledger.main sync <dir>`: reads `<dir>/feed.json` + optional
`<dir>/cache.json`, writes `ledger.json` (`{"items": [...]}`) +
`ledger.csv` (header `id,name,price,stock`) into the cwd, prints exactly
`Ledger: N items (fresh X, cached Y)` with `fresh = len(network)`.
Any other argv → `usage: python -m ledger.main sync <dir>` on **stderr**,
exit **1**, no files written.
`ledger/tests/test_merge.py` holds exactly `test_merge_keeps_cached_only`,
`test_merge_network_wins`, `test_csv_roundtrip` (all green).
`requirements.txt` is exactly `pytest` + newline; `.gitignore` covers
envs, caches, and data files; `README.md` names the entry command
`python -m ledger.main sync <dir>`.

**Constraints**
- **One positional form (`sync <dir>`) — no `--flags`; `newline=""` + `encoding="utf-8"` on every CSV handle.**
- Test 1: feed (4 items) + cache (stale-overlap id 3 + offline-only id
  5) → merged ids `[1, 2, 3, 4, 5]`, `Ledger: 5 items (fresh 4, cached 1)`.
- Test 2: feed only → `Ledger: 4 items (fresh 4, cached 0)`.
- The driver runs the entry with `cwd=tmp` + `PYTHONPATH=<lesson>` and
  your tests with `pytest ledger/tests -q`.

**Coverage**
Package + project-local tests + supporting files + merge contract →
lesson.md Core Concepts 1/3/4/5/6/7; merge + guard idioms unannounced.

**Expected** (Driver cell):
```
Ledger: 5 items (fresh 4, cached 1)
ids: [1, 2, 3, 4, 5]
id,name,price,stock
Ledger: 4 items (fresh 4, cached 0)
usage exit: 1
3 passed
requirements: pytest
readme entry: True
```


In [9]:
import json
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

lesson = Path.cwd()
feed = json.loads((lesson / "ledger_flat" / "feed.json").read_text(encoding="utf-8"))
env = {**os.environ, "PYTHONPATH": str(lesson)}
with tempfile.TemporaryDirectory() as td:
    feeddir = Path(td) / "feed"
    feeddir.mkdir()
    (feeddir / "feed.json").write_text(json.dumps(feed), encoding="utf-8")
    (feeddir / "cache.json").write_text(
        json.dumps({"items": [
            {"id": 3, "name": "muffin", "price": 0.99, "stock": 99},
            {"id": 5, "name": "donut", "price": 1.0, "stock": 3},
        ]}),
        encoding="utf-8",
    )
    work = Path(td) / "work"
    work.mkdir()
    r1 = subprocess.run(
        [sys.executable, "-m", "ledger.main", "sync", str(feeddir)],
        cwd=work, capture_output=True, text=True, env=env,
    )
    print(r1.stdout.strip())
    merged = json.loads((work / "ledger.json").read_text(encoding="utf-8"))
    print("ids:", [it["id"] for it in merged["items"]])
    print((work / "ledger.csv").read_text(encoding="utf-8").splitlines()[0])
with tempfile.TemporaryDirectory() as td:
    feeddir = Path(td) / "feed"
    feeddir.mkdir()
    (feeddir / "feed.json").write_text(json.dumps(feed), encoding="utf-8")
    work = Path(td) / "work"
    work.mkdir()
    r2 = subprocess.run(
        [sys.executable, "-m", "ledger.main", "sync", str(feeddir)],
        cwd=work, capture_output=True, text=True, env=env,
    )
    print(r2.stdout.strip())
with tempfile.TemporaryDirectory() as td:
    r3 = subprocess.run(
        [sys.executable, "-m", "ledger.main"],
        cwd=td, capture_output=True, text=True, env=env,
    )
    print("usage exit:", r3.returncode)
r4 = subprocess.run(
    [sys.executable, "-m", "pytest", "ledger/tests", "-q", "-p", "no:cacheprovider"],
    capture_output=True, text=True, cwd=lesson,
)
_m = re.search(r"(\d+ passed)", r4.stdout + r4.stderr)
print(_m.group(1) if _m else (r4.stdout + r4.stderr)[-300:])
print("requirements:", (lesson / "ledger" / "requirements.txt").read_text(encoding="utf-8").strip())
print("readme entry:", "python -m ledger.main sync <dir>" in (lesson / "ledger" / "README.md").read_text(encoding="utf-8"))

Ledger: 5 items (fresh 4, cached 1)
ids: [1, 2, 3, 4, 5]
id,name,price,stock
Ledger: 4 items (fresh 4, cached 0)
usage exit: 1
3 passed
requirements: pytest
readme entry: True
